# 06 — NDMM PBMC Subset UMAPs & Ranked Genes
### By [Mansi Singh](mansi.singh@alleninstitute.org), Comp Bio, Allen Institute for Immunology


**Aim:** Generate per-cell-type UMAP and ranked-gene visualizations for each L3 subset to guide manual doublet cluster identification. Produces individual PNGs and a combined PDF summary.

## 1 Setup

Subset the dataset by `tidy.aifi_l3` labels. For each cell type:
1. Run Leiden clustering and plot UMAPs
2. Compute ranked marker genes per cluster
3. Overlay doublet scores, key marker genes, and batch covariates on UMAPs
4. Combine into a single PDF for manual review

In [ ]:
%config Completer.use_jedi = False

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import os
import re
import glob
import math
import shutil
from datetime import date

import pandas as pd              # DataFrames
import numpy as np               # Numerical computation
import scanpy as sc              # Single-cell analysis
import matplotlib.pyplot as plt  # Static plotting
from PIL import Image, ImageDraw # Image composition for summary PDF

### 1.1 Helper Functions

In [ ]:
def get_filepaths_with_glob(root_path: str, file_regex: str):
    """Return list of file paths matching a glob pattern in root_path."""
    return glob.glob(os.path.join(root_path, file_regex))

In [ ]:
def extract_celltype(path):
    """Extract cell-type name from harmony-processed h5ad filename."""
    pattern = r'-ndmm-pbmc-(.*?)-celltype-harmony-processed\.h5ad'
    match = re.search(pattern, path)
    if match:
        return match.group(1)
    return None

In [ ]:
def plot_rank_genes(adata):
    """
    Scatter plot of top-20 ranked genes per Leiden cluster.

    Parameters
    ----------
    adata : AnnData
        Must have rank_genes_groups result stored (t-test).

    Returns
    -------
    plt : matplotlib.pyplot module (for saving/display).
    """
    df = sc.get.rank_genes_groups_df(adata, group=None)
    df = df.groupby('group').head(20).reset_index(drop=True)
    groups = df.groupby('group')

    n_groups = len(groups)
    n_cols = 10
    n_rows = math.ceil(n_groups / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))
    fig.suptitle(adata.obs['AIFI_PBMC-Flex_L3'][0], fontsize=50)

    if n_rows == 1:
        axs = axs.reshape(1, -1)

    for ax in axs.flat:
        ax.set_visible(False)

    for i, (name, group) in enumerate(groups):
        row, col = divmod(i, n_cols)
        ax = axs[row, col]
        ax.set_visible(True)
        ax.scatter(group['scores'], group['names'])
        ax.invert_yaxis()
        ax.set_title(str(name) + " vs Rest")
        ax.set_xlabel('T-Statistics')
        ax.set_ylabel('Gene')

    plt.tight_layout()
    return plt

## 2 Generate Summary UMAPs for Doublet Review

### 2.1 Output Paths

In [ ]:
# Output directories for UMAP images
umap_only_dir = '../../../data/rna/ndmm-pbmc-celltypes-umaps-harmony/'
os.makedirs(umap_only_dir, exist_ok=True)

umap_ranked_dir = '../../../data/rna/ndmm-pbmc-celltypes-doublet-qc-umap-rank-genes-harmony/'
os.makedirs(umap_ranked_dir, exist_ok=True)

### 2.2 Load Harmony-Processed Files

In [ ]:
# Path to per-cell-type Harmony-processed objects (produced in NB05)
output_path = "../../../data/rna/ndmm-pbmc-celltypes/"

In [ ]:
# Collect all Harmony-processed h5ad files
filenames = get_filepaths_with_glob(output_path, "*harmony-processed.h5ad")
print(f"Found {len(filenames)} harmony-processed files")

In [ ]:
# Map cell-type names to file paths
cell_types = [extract_celltype(path) for path in filenames]
file_dict = dict(zip(cell_types, filenames))
print(f"{len(file_dict)} cell types to visualize")

['gzmb+vd2gdt',
 'transitionalbcell',
 'gzmk-cd56dimnkcell',
 'adaptivenkcell',
 'isg+cd56dimnkcell',
 'gzmk+cd27+emcd8tcell',
 'clpcell',
 'baeomapcell',
 'il1b+cd14monocyte',
 'cd8mait',
 'c1q+cd16monocyte',
 'proliferatingtcell',
 'corememorybcell',
 'isg+memorycd8tcell',
 'isg+cdc2',
 'dntcell',
 'isg+cd14monocyte',
 'isg+naivecd8tcell',
 'cd56brightnkcell',
 'memorycd4treg',
 'klrf1-gzmb+cd27-memorycd4tcell',
 'pdc',
 'ilc',
 'platelet',
 'corenaivecd4tcell',
 'klrf1-effectorvd1gdt',
 'gzmk+cd56dimnkcell',
 'gzmk+vd2gdt',
 'sox4+naivecd8tcell',
 'gzmb-cd27+emcd4tcell',
 'gzmb-cd27-emcd4tcell',
 'erythrocyte',
 'cd14+cdc2',
 'cmpcell',
 'corenaivebcell',
 'sox4+vd1gdt',
 'klrf1+effectorvd1gdt',
 'isg+naivebcell',
 'plasmacell',
 'cd4mait',
 'corecd14monocyte',
 'gzmk+memorycd4treg',
 'isg+cd16monocyte',
 'corenaivecd8tcell',
 'sox4+naivecd4tcell',
 'naivevd1gdt',
 'isg+mait',
 'klrf1-gzmb+cd27-emcd8tcell',
 'cd27+effectorbcell',
 'cmcd4tcell',
 'hla-drhicdc2',
 'klrb1+memorycd4treg

In [ ]:
# Generate Leiden UMAP for each cell type (skip if output already exists)
for label, file in file_dict.items():
    adata = sc.read_h5ad(file)
    print(adata)
    cell_type_label = adata.obs['tidy.aifi_l3'][0]
    output_file = os.path.join(umap_only_dir, f"{cell_type_label}.png")


    if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
        continue  # Already generated

    print(f"Generating UMAP for: {label}")

    # Save UMAP to scanpy's default figures/ dir, then move to target
    sc.pl.umap(adata, color=['leiden'],
               legend_loc='on data', use_raw=False,
               save='_UMAP.png', title=label)

    umap_img_path = 'figures/umap_UMAP.png'
    if os.path.exists(umap_img_path):
        os.rename(umap_img_path, output_file)

In [ ]:
# Generate multi-panel UMAPs + ranked gene scatter for each cell type
# Marker genes shown: S100A9 (monocyte), PF4 (platelet), CD3E (T cell),
#                     HBB (erythrocyte), MS4A1 (B cell) — used to identify doublet clusters
for label, file in file_dict.items():
    print("Key:", label)
    print("Value:", file)
    
    # Read the h5ad file
    adata = sc.read_h5ad(file)
    print(adata)
    cell_type_label = adata.obs['tidy.aifi_l3'][0]
    output_file = os.path.join(umap_ranked_dir, f"{cell_type_label}.png")

    if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
        continue  # Already generated

    print(f"Processing: {label}")

    # Ranked genes per Leiden cluster
    sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

    # Multi-panel UMAP: doublet score, key markers, metadata covariates
    sc.pl.umap(adata, color=[
        'doublet_score', 'S100A9', 'leiden',
        'PF4', 'CD3E', 'tidy.aifi_l2',
        'HBB', 'MS4A1', 'sample.visitName',
        'batch_id', 'subject.biologicalSex'
    ], ncols=3, save='_UMAP.png')

    # Top-20 ranked genes scatter plot per cluster
    df = sc.get.rank_genes_groups_df(adata, group=None)
    df = df.groupby('group').head(20).reset_index(drop=True)
    groups = df.groupby('group')

    n_groups = len(groups)
    n_cols = 10
    n_rows = math.ceil(n_groups / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))
    fig.suptitle(cell_type_label, fontsize=50)

    if n_rows == 1:
        axs = axs.reshape(1, -1)

    for ax in axs.flat:
        ax.set_visible(False)

    for i, (name, group) in enumerate(groups):
        row, col = divmod(i, n_cols)
        ax = axs[row, col]
        ax.set_visible(True)
        ax.scatter(group['scores'], group['names'])
        ax.invert_yaxis()
        ax.set_title(str(name) + " vs Rest")
        ax.set_xlabel('T-Statistics')
        ax.set_ylabel('Gene')

    plt.tight_layout()
    plt.savefig('scatter_plots.png')

    # Combine UMAP + scatter into a single image
    img2 = Image.open('figures/umap_UMAP.png')
    img1 = Image.open('scatter_plots.png')

    total_width = max(img1.width, img2.width)
    font_size = 20
    title_height = font_size * 2
    total_height = img1.height + img2.height + title_height

    combined_img = Image.new('RGB', (total_width, total_height), 'white')
    combined_img.paste(img1, (0, title_height))
    combined_img.paste(img2, (0, title_height + img1.height))
    combined_img.save(output_file)

In [ ]:
# Clean up any Jupyter checkpoint artifacts
checkpoints_dir = os.path.join(umap_ranked_dir, '.ipynb_checkpoints')
if os.path.isdir(checkpoints_dir):
    shutil.rmtree(checkpoints_dir)
    print(f"Removed: {checkpoints_dir}")

In [ ]:
def pngs_to_pdf(png_files, output_pdf):
    """Merge a list of PNGs into a single multi-page PDF."""
    if not png_files:
        print("No PNG files to convert.")
        return
    images = [Image.open(png).convert('RGBA') for png in png_files]
    rgb_images = [Image.new("RGB", img.size, (255, 255, 255)) for img in images]
    for rgb_img, img in zip(rgb_images, images):
        rgb_img.paste(img, mask=img.split()[3])
    rgb_images[0].save(output_pdf, save_all=True, append_images=rgb_images[1:])


# Combine all per-cell-type images into a single PDF for review
files = sorted(os.listdir(umap_ranked_dir))
png_files = [os.path.join(umap_ranked_dir, f) for f in files if f.endswith('.png')]
output_pdf = '../../../data/rna/ndmm-doublet-qc-output-all-celltypes-l3.pdf'
pngs_to_pdf(png_files, output_pdf)
print(f"Saved combined PDF: {output_pdf}")